# Risk Agent

Looks through the SAP article chunks and pulls out business risks
(competitive threats, regulatory issues, negative sentiment, supply chain problems).

Same setup as the opportunity agent — FAISS index from `embedding_v2.ipynb`,
local Ollama model for reasoning.


In [ ]:
import json
import os
import time

import faiss
import pandas as pd
import requests
from sentence_transformers import SentenceTransformer
from pathlib import Path

cwd = Path.cwd()
DATA_DIR = next((p for p in [cwd / "notebook" / "data", cwd.parent / "notebook" / "data"] if p.exists()), cwd)

CHUNKS_PATH = DATA_DIR / "chunked_data.json"
INDEX_PATH  = DATA_DIR / "sap_intelligence.index"
OUT_PATH    = DATA_DIR / "risks.json"

EMBED_MODEL  = "BAAI/bge-small-en-v1.5"
OLLAMA_HOST  = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "phi4-mini")
TIMEOUT      = 300

print("data dir:", DATA_DIR)


In [ ]:
chunks_df = pd.read_json(CHUNKS_PATH)
index = faiss.read_index(str(INDEX_PATH))
model = SentenceTransformer(EMBED_MODEL)

print("chunks loaded:", len(chunks_df))
print("vectors in index:", index.ntotal)


In [ ]:
def retrieve(query, k=8):
    q_vec = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q_vec, k)
    results = chunks_df.iloc[idx[0]]
    return results["text"].tolist()


def ask_llm(prompt):
    resp = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
        timeout=TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()["response"]


def warmup():
    print("warming up model...")
    start = time.time()
    ask_llm("hi")
    print("warmup done in", round(time.time() - start, 1), "sec")


warmup()


## Build context

In [ ]:
queries = [
    "SAP competitive threats and competitors",
    "SAP regulatory and compliance issues",
    "SAP negative news or criticism",
]

chunks = []
for q in queries:
    chunks.extend(retrieve(q, k=6))

chunks = list(dict.fromkeys(chunks))

context = "\n\n".join(chunks)
print("number of chunks used:", len(chunks))
print("context length (chars):", len(context))


## Risk agent

Returns a JSON list matching the Risk Monitor dashboard fields:
`title`, `risk_category`, `severity_level`, `evidence`, `confidence_score`.


In [ ]:
def risk_agent(context):
    prompt = f"""You are a business risk analyst for SAP.

Read the news context below and identify 3 to 5 concrete business risks.
Risks can be competitive threats, regulatory changes, negative sentiment,
or supply chain issues.

Return ONLY a JSON array, no extra text, no markdown. Each item must look like this:

{{
  "title": "short risk title",
  "risk_category": "Competitive, Regulatory, Sentiment, or Supply Chain",
  "severity_level": "High, Medium, or Low",
  "evidence": "one sentence quoting or summarising the supporting news",
  "confidence_score": a number from 0 to 100
}}

Context:
{context}
"""
    raw = ask_llm(prompt)
    return raw


raw_output = risk_agent(context)
print(raw_output)


## Clean up the JSON

In [ ]:
def parse_json_list(raw_text):
    text = raw_text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text.replace("json", "", 1).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("could not parse JSON, here is the raw text:")
        print(text)
        return []


risks = parse_json_list(raw_output)
print("parsed", len(risks), "risks")
for r in risks:
    print("-", r.get("title"), "|", r.get("risk_category"), "|", r.get("severity_level"))


## Save results for the dashboard

In [ ]:
with open(OUT_PATH, "w") as f:
    json.dump(risks, f, indent=2)

print("saved to", OUT_PATH)
